## Cell 1: Load Data & Initial Inspection

Before any feature engineering or modeling, we inspect the raw dataset — shape, 
data types, missing values, target distribution, and duplicates — to understand 
what we're working with and avoid blind assumptions.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)

SEED = 42
np.random.seed(SEED)

TRAIN_PATH = '/kaggle/input/competitions/playground-series-s6e8/train.csv'
TEST_PATH  = '/kaggle/input/competitions/playground-series-s6e8/test.csv'
SUB_PATH   = '/kaggle/input/competitions/playground-series-s6e8/sample_submission.csv'

train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)
sample_sub = pd.read_csv(SUB_PATH)

TARGET = 'addicted_label'
ID_COL = 'id'

print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")

print("\n--- Data types ---")
print(train.dtypes)

print("\n--- Missing values (train) ---")
print(train.isnull().sum().sort_values(ascending=False))

print("\n--- Missing values (test) ---")
print(test.isnull().sum().sort_values(ascending=False))

print("\n--- Target distribution ---")
print(train[TARGET].value_counts(normalize=True))

print("\n--- Duplicate rows in train ---")
print(train.duplicated().sum())

print("\n--- Duplicate ids ---")
print(f"Train duplicate ids: {train[ID_COL].duplicated().sum()}")
print(f"Test duplicate ids: {test[ID_COL].duplicated().sum()}")

print("\n--- Basic stats (numeric columns) ---")
print(train.describe().T)

print("\n--- Sample rows ---")
print(train.head())

## Cell 2: Feature Engineering

Every feature has missing values (4%–19%), so we treat missingness itself as 
signal — not just something to impute away. We add per-column missing flags, 
a total-missing count, and behavioral ratio/interaction features that capture 
patterns a raw hours-count can't (e.g. what fraction of screen time is social 
media vs gaming, weekend vs weekday drift, notification load per app open).

In [ ]:
FEATURES_NUM = ['age', 'daily_screen_time_hours', 'social_media_hours', 'gaming_hours',
                 'work_study_hours', 'sleep_hours', 'notifications_per_day',
                 'app_opens_per_day', 'weekend_screen_time']
FEATURES_CAT = ['gender', 'stress_level', 'academic_work_impact']

def feature_engineer(df):
    df = df.copy()

    # Missingness as signal
    for c in FEATURES_NUM + FEATURES_CAT:
        df[c + '_missing'] = df[c].isna().astype(int)
    df['total_missing'] = df[[c + '_missing' for c in FEATURES_NUM + FEATURES_CAT]].sum(axis=1)

    # Behavioral ratios (guard against div-by-zero with +0.01)
    df['screen_per_age'] = df['daily_screen_time_hours'] / (df['age'] + 1)
    df['social_ratio'] = df['social_media_hours'] / (df['daily_screen_time_hours'] + 0.01)
    df['gaming_ratio'] = df['gaming_hours'] / (df['daily_screen_time_hours'] + 0.01)
    df['work_study_ratio'] = df['work_study_hours'] / (df['daily_screen_time_hours'] + 0.01)
    df['screen_sleep_ratio'] = df['daily_screen_time_hours'] / (df['sleep_hours'] + 0.01)
    df['weekend_vs_weekday'] = df['weekend_screen_time'] - df['daily_screen_time_hours']
    df['notif_per_open'] = df['notifications_per_day'] / (df['app_opens_per_day'] + 1)
    df['leisure_hours'] = df['social_media_hours'] + df['gaming_hours']
    df['leisure_vs_work'] = df['leisure_hours'] / (df['work_study_hours'] + 0.01)
    df['sleep_deficit'] = 24 - df['sleep_hours'] - df['work_study_hours'] - df['daily_screen_time_hours']

    # Encode ordinal / binary categoricals
    stress_map = {'Low': 0, 'Medium': 1, 'High': 2}
    df['stress_level_enc'] = df['stress_level'].map(stress_map)
    df['academic_work_impact_enc'] = df['academic_work_impact'].map({'No': 0, 'Yes': 1})
    df['gender'] = df['gender'].astype('category')

    return df

train_fe = feature_engineer(train)
test_fe  = feature_engineer(test)

ENG_NUM = ['screen_per_age', 'social_ratio', 'gaming_ratio', 'work_study_ratio',
           'screen_sleep_ratio', 'weekend_vs_weekday', 'notif_per_open',
           'leisure_hours', 'leisure_vs_work', 'sleep_deficit']
MISS_FLAGS = [c + '_missing' for c in FEATURES_NUM + FEATURES_CAT]

ALL_FEATURES = FEATURES_NUM + ENG_NUM + MISS_FLAGS + \
                ['total_missing', 'stress_level_enc', 'academic_work_impact_enc', 'gender']

X = train_fe[ALL_FEATURES]
y = train_fe[TARGET]
X_test = test_fe[ALL_FEATURES]

print(f"Total features: {len(ALL_FEATURES)}")
print(X.isnull().sum().sort_values(ascending=False).head(10))
print(X.shape, X_test.shape)

## Cell 3: Baseline Multi-Model Training (LightGBM, XGBoost, CatBoost) — GPU

We train three gradient-boosted tree models with 5-fold Stratified K-Fold and 
early stopping on each fold's validation split, using GPU acceleration. This 
lets the trees handle missing values natively, keeps each model's growth in 
check (early stopping halts before it starts memorizing noise), and gives us 
an out-of-fold (OOF) AUC per model — a leakage-free estimate of real 
generalization before we tune or ensemble anything.

In [ ]:
import time

oof_lgb = np.zeros(len(X)); oof_xgb = np.zeros(len(X)); oof_cat = np.zeros(len(X))
test_lgb = np.zeros(len(X_test)); test_xgb = np.zeros(len(X_test)); test_cat = np.zeros(len(X_test))

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y), start=1):
    print(f"\n=== Fold {fold} starting ===", flush=True)
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
    Xc_tr, Xc_val = X_cat.iloc[tr_idx], X_cat.iloc[val_idx]

    t0 = time.time()
    m_lgb = lgb.LGBMClassifier(**lgb_params)
    m_lgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
              callbacks=[lgb.early_stopping(150, verbose=False), lgb.log_evaluation(0)])
    oof_lgb[val_idx] = m_lgb.predict_proba(X_val)[:, 1]
    test_lgb += m_lgb.predict_proba(X_test)[:, 1] / N_FOLDS
    print(f"  LightGBM done in {time.time()-t0:.1f}s | AUC: {roc_auc_score(y_val, oof_lgb[val_idx]):.5f}", flush=True)

    t0 = time.time()
    m_xgb = xgb.XGBClassifier(**xgb_params, early_stopping_rounds=150)
    m_xgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    oof_xgb[val_idx] = m_xgb.predict_proba(X_val)[:, 1]
    test_xgb += m_xgb.predict_proba(X_test)[:, 1] / N_FOLDS
    print(f"  XGBoost done in {time.time()-t0:.1f}s | AUC: {roc_auc_score(y_val, oof_xgb[val_idx]):.5f}", flush=True)

    t0 = time.time()
    m_cat = CatBoostClassifier(**cat_params)
    m_cat.fit(Xc_tr, y_tr, eval_set=(Xc_val, y_val), cat_features=cat_idx,
              early_stopping_rounds=150, use_best_model=True)
    oof_cat[val_idx] = m_cat.predict_proba(Xc_val)[:, 1]
    test_cat += m_cat.predict_proba(X_test_cat)[:, 1] / N_FOLDS
    print(f"  CatBoost done in {time.time()-t0:.1f}s | AUC: {roc_auc_score(y_val, oof_cat[val_idx]):.5f}", flush=True)

print(f"\nOOF AUC — LightGBM: {roc_auc_score(y, oof_lgb):.5f}")
print(f"OOF AUC — XGBoost:  {roc_auc_score(y, oof_xgb):.5f}")
print(f"OOF AUC — CatBoost: {roc_auc_score(y, oof_cat):.5f}")

In [ ]:
# Quick sanity test with far fewer estimators, 1 fold only
test_lgb_params = {**lgb_params, 'n_estimators': 100}
test_xgb_params = {**xgb_params, 'n_estimators': 100}
test_cat_params = {**cat_params, 'iterations': 100}

from sklearn.model_selection import train_test_split
X_tr_s, X_val_s, y_tr_s, y_val_s = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
Xc_tr_s, Xc_val_s = X_cat.loc[X_tr_s.index], X_cat.loc[X_val_s.index]

import time

t0 = time.time()
m = lgb.LGBMClassifier(**test_lgb_params)
m.fit(X_tr_s, y_tr_s, eval_set=[(X_val_s, y_val_s)], callbacks=[lgb.log_evaluation(0)])
print(f"LightGBM done in {time.time()-t0:.1f}s")

t0 = time.time()
m = xgb.XGBClassifier(**test_xgb_params)
m.fit(X_tr_s, y_tr_s, eval_set=[(X_val_s, y_val_s)], verbose=False)
print(f"XGBoost done in {time.time()-t0:.1f}s")

t0 = time.time()
m = CatBoostClassifier(**test_cat_params)
m.fit(Xc_tr_s, y_tr_s, eval_set=(Xc_val_s, y_val_s), cat_features=cat_idx)
print(f"CatBoost done in {time.time()-t0:.1f}s")

## Cell 4: Fold-Safe Target Encoding + Behavioral Clustering

We add two more feature types: (1) target-encoded categoricals, computed 
strictly inside each fold using only that fold's training data — never the 
full dataset — to avoid leakage; and (2) KMeans clusters over the behavioral 
features, giving the models a compressed "usage pattern" signal that raw 
numeric columns don't directly expose. Both are added as new OOF-safe columns 
on top of the existing feature set.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler as SS

# --- KMeans behavioral clustering (fit once on train, safe since it's unsupervised — no target used) ---
cluster_cols = ['daily_screen_time_hours', 'social_media_hours', 'gaming_hours',
                 'work_study_hours', 'sleep_hours', 'weekend_screen_time']

X_cluster_train = train_fe[cluster_cols].fillna(train_fe[cluster_cols].median())
X_cluster_test  = test_fe[cluster_cols].fillna(train_fe[cluster_cols].median())

cluster_scaler = SS()
X_cluster_train_scaled = cluster_scaler.fit_transform(X_cluster_train)
X_cluster_test_scaled  = cluster_scaler.transform(X_cluster_test)

N_CLUSTERS = 8
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=SEED, n_init=10)
train_fe['behavior_cluster'] = kmeans.fit_predict(X_cluster_train_scaled)
test_fe['behavior_cluster']  = kmeans.predict(X_cluster_test_scaled)
train_fe['behavior_cluster'] = train_fe['behavior_cluster'].astype('category')
test_fe['behavior_cluster']  = test_fe['behavior_cluster'].astype('category')

# Distance to own cluster centroid — captures "how typical" a user is for their group
train_fe['dist_to_centroid'] = kmeans.transform(X_cluster_train_scaled).min(axis=1)
test_fe['dist_to_centroid']  = kmeans.transform(X_cluster_test_scaled).min(axis=1)

# --- Fold-safe target encoding ---
TE_COLS = ['gender', 'stress_level', 'academic_work_impact']

def add_target_encoding(train_df, test_df, y_full, cols, n_splits=5, smoothing=20):
    train_te = pd.DataFrame(index=train_df.index)
    test_te = pd.DataFrame(index=test_df.index)
    global_mean = y_full.mean()

    skf_te = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)

    for col in cols:
        train_te[col + '_te'] = np.nan
        test_pred_accum = np.zeros(len(test_df))

        for tr_idx, val_idx in skf_te.split(train_df, y_full):
            fold_train = train_df.iloc[tr_idx].copy()
            fold_train['_y'] = y_full.iloc[tr_idx].values
            fold_train[col] = fold_train[col].astype(str)   # force plain string
            stats = fold_train.groupby(col)['_y'].agg(['mean', 'count'])
            smoothed = (stats['mean'] * stats['count'] + global_mean * smoothing) / (stats['count'] + smoothing)

            val_col = train_df.iloc[val_idx][col].astype(str)   # force plain string
            train_te.iloc[val_idx, train_te.columns.get_loc(col + '_te')] = val_col.map(smoothed).fillna(global_mean).astype(float).values

            test_pred_accum += test_df[col].astype(str).map(smoothed).fillna(global_mean).astype(float).values / n_splits

        test_te[col + '_te'] = test_pred_accum

    return train_te, test_te

train_te, test_te = add_target_encoding(train_fe, test_fe, y, TE_COLS)
train_fe = pd.concat([train_fe, train_te], axis=1)
test_fe = pd.concat([test_fe, test_te], axis=1)

# --- Update feature list ---
NEW_FEATURES = ['behavior_cluster', 'dist_to_centroid'] + [c + '_te' for c in TE_COLS]
ALL_FEATURES_V2 = ALL_FEATURES + NEW_FEATURES

X = train_fe[ALL_FEATURES_V2]
y = train_fe[TARGET]
X_test = test_fe[ALL_FEATURES_V2]

print(f"Total features now: {len(ALL_FEATURES_V2)}")
print(f"New features: {NEW_FEATURES}")
print(X[NEW_FEATURES].describe())
print(X[NEW_FEATURES].isnull().sum())

## Cell 5: Retrain on Expanded Feature Set (Clustering + Target Encoding)

Same models, same fold setup as Cell 3 — only the feature set changed (35 → 
40 columns, adding behavior_cluster, dist_to_centroid, and 3 target-encoded 
categoricals). Comparing OOF AUC directly against the Cell 3 baseline tells 
us whether these new features are actually earning their place, since more 
features isn't automatically better.

In [ ]:
oof_lgb_v2 = np.zeros(len(X)); oof_xgb_v2 = np.zeros(len(X)); oof_cat_v2 = np.zeros(len(X))
test_lgb_v2 = np.zeros(len(X_test)); test_xgb_v2 = np.zeros(len(X_test)); test_cat_v2 = np.zeros(len(X_test))

X_cat_v2 = X.copy()
X_cat_v2['gender'] = X_cat_v2['gender'].astype(str)
X_cat_v2['behavior_cluster'] = X_cat_v2['behavior_cluster'].astype(str)

X_test_cat_v2 = X_test.copy()
X_test_cat_v2['gender'] = X_test_cat_v2['gender'].astype(str)
X_test_cat_v2['behavior_cluster'] = X_test_cat_v2['behavior_cluster'].astype(str)

cat_idx_v2 = [X_cat_v2.columns.get_loc('gender'), X_cat_v2.columns.get_loc('behavior_cluster')]

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y), start=1):
    print(f"\n=== Fold {fold} starting ===", flush=True)
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
    Xc_tr, Xc_val = X_cat_v2.iloc[tr_idx], X_cat_v2.iloc[val_idx]

    t0 = time.time()
    m_lgb = lgb.LGBMClassifier(**lgb_params)
    m_lgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
              callbacks=[lgb.early_stopping(150, verbose=False), lgb.log_evaluation(0)])
    oof_lgb_v2[val_idx] = m_lgb.predict_proba(X_val)[:, 1]
    test_lgb_v2 += m_lgb.predict_proba(X_test)[:, 1] / N_FOLDS
    print(f"  LightGBM done in {time.time()-t0:.1f}s | AUC: {roc_auc_score(y_val, oof_lgb_v2[val_idx]):.5f}", flush=True)

    t0 = time.time()
    m_xgb = xgb.XGBClassifier(**xgb_params, early_stopping_rounds=150)
    m_xgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    oof_xgb_v2[val_idx] = m_xgb.predict_proba(X_val)[:, 1]
    test_xgb_v2 += m_xgb.predict_proba(X_test)[:, 1] / N_FOLDS
    print(f"  XGBoost done in {time.time()-t0:.1f}s | AUC: {roc_auc_score(y_val, oof_xgb_v2[val_idx]):.5f}", flush=True)

    t0 = time.time()
    m_cat = CatBoostClassifier(**cat_params)
    m_cat.fit(Xc_tr, y_tr, eval_set=(Xc_val, y_val), cat_features=cat_idx_v2,
              early_stopping_rounds=150, use_best_model=True)
    oof_cat_v2[val_idx] = m_cat.predict_proba(Xc_val)[:, 1]
    test_cat_v2 += m_cat.predict_proba(X_test_cat_v2)[:, 1] / N_FOLDS
    print(f"  CatBoost done in {time.time()-t0:.1f}s | AUC: {roc_auc_score(y_val, oof_cat_v2[val_idx]):.5f}", flush=True)

print(f"\n=== v2 (expanded features) ===")
print(f"OOF AUC — LightGBM: {roc_auc_score(y, oof_lgb_v2):.5f}  (baseline: 0.96389)")
print(f"OOF AUC — XGBoost:  {roc_auc_score(y, oof_xgb_v2):.5f}  (baseline: 0.96421)")
print(f"OOF AUC — CatBoost: {roc_auc_score(y, oof_cat_v2):.5f}  (baseline: 0.95790)")

## Cell 6: Stacked Meta-Model (Level 2 Logistic Regression)

Since v2's features (clustering + target encoding) didn't beat baseline, we go 
back to the original 35-feature set and instead stack the three baseline models' 
out-of-fold predictions (oof_lgb, oof_xgb, oof_cat) as inputs to a Level 2 
meta-model. A simple, well-regularized Logistic Regression on top of OOF 
predictions typically generalizes better than manual averaging, because it 
learns the actual relative reliability of each base model instead of assuming 
equal weight — while staying leakage-safe since it's trained only on OOF data.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict

# Stack the baseline (v1, 35-feature) OOF predictions as meta-features
meta_X_train = np.column_stack([oof_lgb, oof_xgb, oof_cat])
meta_X_test  = np.column_stack([test_lgb, test_xgb, test_cat])

print("Correlation between base model OOF predictions:")
print(pd.DataFrame(meta_X_train, columns=['lgb', 'xgb', 'cat']).corr())

# Meta-model: get its own OOF AUC via cross-validation to keep this leakage-safe
meta_model = LogisticRegression(C=1.0, max_iter=1000)

skf_meta = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
meta_oof = cross_val_predict(
    meta_model, meta_X_train, y, cv=skf_meta, method='predict_proba'
)[:, 1]

meta_stack_auc = roc_auc_score(y, meta_oof)
print(f"\nStacked meta-model OOF AUC: {meta_stack_auc:.5f}")
print(f"vs individual — LGB: {roc_auc_score(y, oof_lgb):.5f} | XGB: {roc_auc_score(y, oof_xgb):.5f} | CAT: {roc_auc_score(y, oof_cat):.5f}")

# Simple weighted average as a sanity-check comparison
simple_avg = (oof_lgb + oof_xgb + oof_cat) / 3
print(f"Simple average OOF AUC: {roc_auc_score(y, simple_avg):.5f}")

# Fit meta-model on full OOF data, predict on test meta-features
meta_model.fit(meta_X_train, y)
print(f"\nMeta-model learned weights: LGB={meta_model.coef_[0][0]:.3f}, XGB={meta_model.coef_[0][1]:.3f}, CAT={meta_model.coef_[0][2]:.3f}")

test_meta_pred = meta_model.predict_proba(meta_X_test)[:, 1]

## Cell 7: Neural Network (Architectural Diversity for Stacking)

The three tree models are 98-99.6% correlated with each other — they're 
finding the same patterns, so stacking them adds almost nothing. A neural 
network learns fundamentally different decision boundaries than trees, so 
even if its standalone AUC is similar or slightly lower, it can genuinely 
improve the stack by being wrong in different places than the trees are. 
Trained with the same 5-fold OOF setup to stay leakage-safe and comparable.

In [ ]:
import torch
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.get_device_name(0))
print(torch.cuda.get_device_capability(0))

In [ ]:
device = torch.device('cpu')
print(f"Using device: {device}")

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

device = torch.device('cpu')
print(f"Using device: {device}")

# NN needs fully numeric, scaled, NaN-free input — one-hot encode gender/behavior_cluster
NN_FEATURES = ALL_FEATURES.copy()
X_nn_base = train_fe[NN_FEATURES].copy()
X_nn_test_base = test_fe[NN_FEATURES].copy()

gender_dummies_tr = pd.get_dummies(X_nn_base['gender'], prefix='gender')
gender_dummies_te = pd.get_dummies(X_nn_test_base['gender'], prefix='gender')
gender_dummies_te = gender_dummies_te.reindex(columns=gender_dummies_tr.columns, fill_value=0)

X_nn_base = pd.concat([X_nn_base.drop(columns=['gender']), gender_dummies_tr], axis=1)
X_nn_test_base = pd.concat([X_nn_test_base.drop(columns=['gender']), gender_dummies_te], axis=1)

class ResidualBlock(nn.Module):
    def __init__(self, dim, dropout):
        super().__init__()
        self.fc1 = nn.Linear(dim, dim)
        self.fc2 = nn.Linear(dim, dim)
        self.bn = nn.BatchNorm1d(dim)
        self.act = nn.ReLU()
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        res = x
        x = self.act(self.fc1(x))
        x = self.drop(x)
        x = self.fc2(x)
        x = self.bn(x + res)
        return self.act(x)

class NNClassifier(nn.Module):
    def __init__(self, input_dim, hidden=128, dropout=0.3):
        super().__init__()
        self.inp = nn.Linear(input_dim, hidden)
        self.act = nn.ReLU()
        self.drop = nn.Dropout(dropout)
        self.block1 = ResidualBlock(hidden, dropout)
        self.block2 = ResidualBlock(hidden, dropout)
        self.out = nn.Linear(hidden, 1)
    def forward(self, x):
        x = self.drop(self.act(self.inp(x)))
        x = self.block1(x)
        x = self.block2(x)
        return self.out(x).squeeze(1)

oof_nn = np.zeros(len(X_nn_base))
test_nn = np.zeros(len(X_nn_test_base))

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_nn_base, y), start=1):
    t0 = time.time()
    X_tr_raw, X_val_raw = X_nn_base.iloc[tr_idx], X_nn_base.iloc[val_idx]
    y_tr_f, y_val_f = y.iloc[tr_idx].values, y.iloc[val_idx].values

    imputer = SimpleImputer(strategy='median')
    scaler = StandardScaler()
    X_tr_proc = scaler.fit_transform(imputer.fit_transform(X_tr_raw))
    X_val_proc = scaler.transform(imputer.transform(X_val_raw))
    X_test_proc = scaler.transform(imputer.transform(X_nn_test_base))

    train_ds = TensorDataset(torch.tensor(X_tr_proc, dtype=torch.float32),
                              torch.tensor(y_tr_f, dtype=torch.float32))
    train_loader = DataLoader(train_ds, batch_size=4096, shuffle=True)
    X_val_t = torch.tensor(X_val_proc, dtype=torch.float32).to(device)
    X_test_t = torch.tensor(X_test_proc, dtype=torch.float32).to(device)

    model = NNClassifier(input_dim=X_tr_proc.shape[1]).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-3)
    criterion = nn.BCEWithLogitsLoss()
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

    best_auc, best_state, patience_ctr = -np.inf, None, 0
    for epoch in range(30):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_preds = torch.sigmoid(model(X_val_t)).cpu().numpy()
        val_auc = roc_auc_score(y_val_f, val_preds)
        scheduler.step(val_auc)

        if val_auc > best_auc:
            best_auc, best_state, patience_ctr = val_auc, {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            patience_ctr += 1
            if patience_ctr >= 5:
                break

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        oof_nn[val_idx] = torch.sigmoid(model(X_val_t)).cpu().numpy()
        test_nn += torch.sigmoid(model(X_test_t)).cpu().numpy() / N_FOLDS

    print(f"Fold {fold} | NN done in {time.time()-t0:.1f}s | Best AUC: {best_auc:.5f}", flush=True)

print(f"\nOOF AUC — Neural Net: {roc_auc_score(y, oof_nn):.5f}")
print(f"(For comparison — LGB: {roc_auc_score(y, oof_lgb):.5f} | XGB: {roc_auc_score(y, oof_xgb):.5f} | CAT: {roc_auc_score(y, oof_cat):.5f})")
print(f"\nCorrelation with tree models:")
print(pd.DataFrame({'nn': oof_nn, 'lgb': oof_lgb, 'xgb': oof_xgb, 'cat': oof_cat}).corr())

## Cell 8: Four-Model Stack (LGB + XGB + CAT + NN)

The neural net correlates meaningfully lower with the trees (0.93-0.95) than 
the trees do with each other (0.98-0.996), meaning it's making different kinds 
of errors — the ingredient a stack actually needs. We rebuild the Level 2 
meta-model with all four OOF predictions as inputs, using the same leakage-safe 
cross_val_predict approach as before.

In [ ]:
meta_X_train_v2 = np.column_stack([oof_lgb, oof_xgb, oof_cat, oof_nn])
meta_X_test_v2  = np.column_stack([test_lgb, test_xgb, test_cat, test_nn])

meta_model_v2 = LogisticRegression(C=1.0, max_iter=1000)

skf_meta = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
meta_oof_v2 = cross_val_predict(
    meta_model_v2, meta_X_train_v2, y, cv=skf_meta, method='predict_proba'
)[:, 1]

meta_stack_auc_v2 = roc_auc_score(y, meta_oof_v2)
print(f"4-model stacked OOF AUC: {meta_stack_auc_v2:.5f}")
print(f"vs 3-model stack: 0.96432")
print(f"vs best individual (XGB): {roc_auc_score(y, oof_xgb):.5f}")

meta_model_v2.fit(meta_X_train_v2, y)
print(f"\nMeta-model weights: LGB={meta_model_v2.coef_[0][0]:.3f}, XGB={meta_model_v2.coef_[0][1]:.3f}, "
      f"CAT={meta_model_v2.coef_[0][2]:.3f}, NN={meta_model_v2.coef_[0][3]:.3f}")

test_meta_pred_v2 = meta_model_v2.predict_proba(meta_X_test_v2)[:, 1]

# Try a non-linear meta-model too, since 4 inputs is small enough to experiment cheaply
from sklearn.ensemble import GradientBoostingClassifier
meta_gbm = GradientBoostingClassifier(n_estimators=100, max_depth=2, learning_rate=0.05, random_state=SEED)
meta_oof_gbm = cross_val_predict(meta_gbm, meta_X_train_v2, y, cv=skf_meta, method='predict_proba')[:, 1]
print(f"\nGBM meta-model OOF AUC: {roc_auc_score(y, meta_oof_gbm):.5f}")

In [ ]:
from sklearn.metrics import accuracy_score

# Convert probabilities to hard predictions at 0.5 threshold and check actual accuracy
for name, oof_preds in [('LGB', oof_lgb), ('XGB', oof_xgb), ('CAT', oof_cat), ('NN', oof_nn), ('3-model stack', meta_oof)]:
    hard_preds = (oof_preds >= 0.5).astype(int)
    acc = accuracy_score(y, hard_preds)
    print(f"{name}: Accuracy = {acc:.5f}  (AUC-based, 0.5 threshold)")

# Naive baseline: always predict the majority class
naive_acc = max(y.mean(), 1 - y.mean())
print(f"\nNaive majority-class baseline accuracy: {naive_acc:.5f}")

In [ ]:
from sklearn.metrics import accuracy_score

thresholds = np.arange(0.3, 0.71, 0.02)
best_thresh, best_acc = 0.5, 0
for t in thresholds:
    acc = accuracy_score(y, (meta_oof >= t).astype(int))
    if acc > best_acc:
        best_acc, best_thresh = acc, t
    print(f"Threshold {t:.2f}: Accuracy = {acc:.5f}")

print(f"\nBest threshold: {best_thresh:.2f} | Best accuracy: {best_acc:.5f}")

## Cell 9: Residual Feature + Value-Level Target Encoding + Higher Capacity

Two changes validated by community research on this exact competition:
(1) `other_screen_time` = daily_screen_time - (social_media + gaming + work_study), 
a residual that recovers signal the raw components can't show individually, and 
(2) value-level target encoding on numeric columns (encoding the exact rounded 
value, not just magnitude) combined with higher model capacity (255 leaves, 1500 
rounds) — worth roughly +0.0046 AUC over a shallow baseline. Both done leakage-safe: 
the residual is pure arithmetic (no target used), and target encoding is computed 
strictly inside each fold.

In [ ]:
# --- Residual feature (pure arithmetic, no leakage risk) ---
train_fe['other_screen_time'] = train_fe['daily_screen_time_hours'] - (
    train_fe['social_media_hours'] + train_fe['gaming_hours'] + train_fe['work_study_hours']
)
test_fe['other_screen_time'] = test_fe['daily_screen_time_hours'] - (
    test_fe['social_media_hours'] + test_fe['gaming_hours'] + test_fe['work_study_hours']
)

# --- Value-level target encoding on rounded numeric columns ---
VLTE_COLS = ['daily_screen_time_hours', 'social_media_hours', 'gaming_hours',
             'work_study_hours', 'weekend_screen_time', 'notifications_per_day',
             'app_opens_per_day', 'sleep_hours', 'age']

def add_value_level_te(train_df, test_df, y_full, cols, n_splits=5, smoothing=30, round_decimals=2):
    train_te = pd.DataFrame(index=train_df.index)
    test_te = pd.DataFrame(index=test_df.index)
    global_mean = y_full.mean()

    skf_te = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)

    for col in cols:
        train_te[col + '_vlte'] = np.nan
        test_pred_accum = np.zeros(len(test_df))

        train_rounded = train_df[col].round(round_decimals)
        test_rounded = test_df[col].round(round_decimals)

        for tr_idx, val_idx in skf_te.split(train_df, y_full):
            fold_vals = train_rounded.iloc[tr_idx]
            fold_y = y_full.iloc[tr_idx].values
            stats = pd.DataFrame({'v': fold_vals.values, 'y': fold_y}).groupby('v')['y'].agg(['mean', 'count'])
            smoothed = (stats['mean'] * stats['count'] + global_mean * smoothing) / (stats['count'] + smoothing)

            val_vals = train_rounded.iloc[val_idx]
            train_te.iloc[val_idx, train_te.columns.get_loc(col + '_vlte')] = \
                val_vals.map(smoothed).fillna(global_mean).astype(float).values

            test_pred_accum += test_rounded.map(smoothed).fillna(global_mean).astype(float).values / n_splits

        test_te[col + '_vlte'] = test_pred_accum

    return train_te, test_te

train_vlte, test_vlte = add_value_level_te(train_fe, test_fe, y, VLTE_COLS)
train_fe = pd.concat([train_fe, train_vlte], axis=1)
test_fe = pd.concat([test_fe, test_vlte], axis=1)

# --- Final feature set: original 35 + residual + value-level TE ---
FINAL_FEATURES = ALL_FEATURES + ['other_screen_time'] + [c + '_vlte' for c in VLTE_COLS]

X = train_fe[FINAL_FEATURES]
y = train_fe[TARGET]
X_test = test_fe[FINAL_FEATURES]

print(f"Total features: {len(FINAL_FEATURES)}")
print(f"other_screen_time stats:\n{X['other_screen_time'].describe()}")
print(f"\nRows where other_screen_time is negative: {(X['other_screen_time'] < 0).sum()}")
print(f"\nNull counts in new columns:")
print(X[['other_screen_time'] + [c + '_vlte' for c in VLTE_COLS]].isnull().sum())

## Cell 10: Final Model — Higher Capacity + Residual + Value-Level Target Encoding

Retraining with the validated changes: other_screen_time (arithmetic residual) and 
value-level target encoding on 9 numeric columns, combined with higher LightGBM 
capacity (255 leaves, 1500→3000 rounds with early stopping) since community research 
on this exact competition found the encoding+capacity combination worth roughly 
+0.0046 AUC over a shallow baseline — the largest validated single lever found.

In [ ]:
lgb_params_v2 = dict(
    objective='binary', metric='auc', boosting_type='gbdt',
    n_estimators=4000, learning_rate=0.02, num_leaves=255,
    min_child_samples=20, subsample=0.8, subsample_freq=1,
    colsample_bytree=0.75, reg_alpha=0.5, reg_lambda=1.0,
    verbosity=-1, random_state=SEED,
    device_type='gpu', gpu_use_dp=True
)

X_cat_final = X.copy()
X_cat_final['gender'] = X_cat_final['gender'].astype(str)
X_test_cat_final = X_test.copy()
X_test_cat_final['gender'] = X_test_cat_final['gender'].astype(str)

oof_lgb_final = np.zeros(len(X))
test_lgb_final = np.zeros(len(X_test))

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y), start=1):
    print(f"\n=== Fold {fold} starting ===", flush=True)
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    t0 = time.time()
    m_lgb = lgb.LGBMClassifier(**lgb_params_v2)
    m_lgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
              callbacks=[lgb.early_stopping(200, verbose=False), lgb.log_evaluation(0)])
    oof_lgb_final[val_idx] = m_lgb.predict_proba(X_val)[:, 1]
    test_lgb_final += m_lgb.predict_proba(X_test)[:, 1] / N_FOLDS
    print(f"  LightGBM (v2 capacity) done in {time.time()-t0:.1f}s | AUC: {roc_auc_score(y_val, oof_lgb_final[val_idx]):.5f}", flush=True)

print(f"\n=== Final LightGBM OOF AUC: {roc_auc_score(y, oof_lgb_final):.5f} ===")
print(f"vs original baseline: 0.96389")
print(f"vs 3-model stack: 0.96432")


=== Fold 1 starting ===
  LightGBM (v2 capacity) done in 221.0s | AUC: 0.96696

=== Fold 2 starting ===
  LightGBM (v2 capacity) done in 193.7s | AUC: 0.96764

=== Fold 3 starting ===
  LightGBM (v2 capacity) done in 195.7s | AUC: 0.96792

=== Fold 4 starting ===
  LightGBM (v2 capacity) done in 202.1s | AUC: 0.96827

=== Fold 5 starting ===


In [27]:
# Fix test predictions — was divided by 5, only 4 folds actually ran
test_lgb_final_fixed = test_lgb_final * (5 / 4)

# Fix OOF AUC — exclude fold 5 rows (they're still zero, never got a prediction)
completed_mask = oof_lgb_final != 0  # rows from folds 1-4 only
oof_auc_4fold = roc_auc_score(y[completed_mask], oof_lgb_final[completed_mask])

print(f"LightGBM AUC (folds 1-4 only, {completed_mask.sum()} rows): {oof_auc_4fold:.5f}")

LightGBM AUC (folds 1-4 only, 691369 rows): 0.96766


In [28]:
test_lgb_final_fixed = test_lgb_final * (5 / 4)

completed_mask = oof_lgb_final != 0
oof_auc_4fold = roc_auc_score(y[completed_mask], oof_lgb_final[completed_mask])

print(f"LightGBM AUC (folds 1-4 only, {completed_mask.sum()} rows): {oof_auc_4fold:.5f}")

LightGBM AUC (folds 1-4 only, 691369 rows): 0.96766


In [29]:
submission = pd.DataFrame({
    'id': test['id'],
    'addicted_label': test_lgb_final
})

submission.to_csv('submission.csv', index=False)
print("Saved: submission.csv")
print(submission.head())
print(f"\nShape: {submission.shape}")
print(f"Prediction range: [{submission['addicted_label'].min():.4f}, {submission['addicted_label'].max():.4f}]")

Saved: submission.csv
       id  addicted_label
0  691369        0.999589
1  691370        0.938246
2  691371        0.875665
3  691372        0.996010
4  691373        0.999046

Shape: (296302, 2)
Prediction range: [0.0005, 1.0000]


In [30]:
# ============ Cell: XGBoost + CatBoost (same 45 features) + Final Stack + Submission ============
import time
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict

xgb_params_v2 = dict(
    objective='binary:logistic', eval_metric='auc',
    n_estimators=4000, learning_rate=0.02, max_depth=10,
    min_child_weight=10, subsample=0.8, colsample_bytree=0.75,
    reg_alpha=0.5, reg_lambda=1.0, enable_categorical=True,
    random_state=SEED, tree_method='hist', device='cuda', verbosity=0
)
cat_params_v2 = dict(
    loss_function='Logloss', eval_metric='AUC',
    iterations=4000, learning_rate=0.02, depth=10,
    l2_leaf_reg=3, random_state=SEED,
    task_type='GPU', logging_level='Silent'
)

X_cat_final = X.copy()
X_cat_final['gender'] = X_cat_final['gender'].astype(str)
X_test_cat_final = X_test.copy()
X_test_cat_final['gender'] = X_test_cat_final['gender'].astype(str)
cat_idx_final = [X_cat_final.columns.get_loc('gender')]

oof_xgb_final = np.zeros(len(X)); oof_cat_final = np.zeros(len(X))
test_xgb_final = np.zeros(len(X_test)); test_cat_final = np.zeros(len(X_test))

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y), start=1):
    print(f"\n=== Fold {fold} starting ===", flush=True)
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
    Xc_tr, Xc_val = X_cat_final.iloc[tr_idx], X_cat_final.iloc[val_idx]

    t0 = time.time()
    m_xgb = xgb.XGBClassifier(**xgb_params_v2, early_stopping_rounds=200)
    m_xgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    oof_xgb_final[val_idx] = m_xgb.predict_proba(X_val)[:, 1]
    test_xgb_final += m_xgb.predict_proba(X_test)[:, 1] / N_FOLDS
    print(f"  XGBoost done in {time.time()-t0:.1f}s | AUC: {roc_auc_score(y_val, oof_xgb_final[val_idx]):.5f}", flush=True)

    t0 = time.time()
    m_cat = CatBoostClassifier(**cat_params_v2)
    m_cat.fit(Xc_tr, y_tr, eval_set=(Xc_val, y_val), cat_features=cat_idx_final,
              early_stopping_rounds=200, use_best_model=True)
    oof_cat_final[val_idx] = m_cat.predict_proba(Xc_val)[:, 1]
    test_cat_final += m_cat.predict_proba(X_test_cat_final)[:, 1] / N_FOLDS
    print(f"  CatBoost done in {time.time()-t0:.1f}s | AUC: {roc_auc_score(y_val, oof_cat_final[val_idx]):.5f}", flush=True)

print(f"\n=== Individual Final OOF AUCs ===")
print(f"LightGBM: {roc_auc_score(y, oof_lgb_final):.5f}")
print(f"XGBoost:  {roc_auc_score(y, oof_xgb_final):.5f}")
print(f"CatBoost: {roc_auc_score(y, oof_cat_final):.5f}")

# ---- Stack all three ----
meta_X_final = np.column_stack([oof_lgb_final, oof_xgb_final, oof_cat_final])
meta_X_test_final = np.column_stack([test_lgb_final, test_xgb_final, test_cat_final])

meta_model_final = LogisticRegression(C=1.0, max_iter=1000)
meta_oof_final = cross_val_predict(meta_model_final, meta_X_final, y, cv=skf, method='predict_proba')[:, 1]

print(f"\n=== FINAL STACKED OOF AUC: {roc_auc_score(y, meta_oof_final):.5f} ===")
print(f"vs LightGBM alone: {roc_auc_score(y, oof_lgb_final):.5f}")

meta_model_final.fit(meta_X_final, y)
final_test_predictions = meta_model_final.predict_proba(meta_X_test_final)[:, 1]

# ---- Save as a NEW file, doesn't overwrite your first submission ----
submission_v2 = pd.DataFrame({
    'id': test['id'],
    'addicted_label': final_test_predictions
})
submission_v2.to_csv('submission_v2.csv', index=False)
print("\nSaved: submission_v2.csv")
print(submission_v2.head())
print(f"Shape: {submission_v2.shape}")


=== Fold 1 starting ===
  XGBoost done in 31.4s | AUC: 0.96705


Default metric period is 5 because AUC is/are not implemented for GPU


  CatBoost done in 286.9s | AUC: 0.96612

=== Fold 2 starting ===
  XGBoost done in 28.6s | AUC: 0.96776


Default metric period is 5 because AUC is/are not implemented for GPU


  CatBoost done in 286.8s | AUC: 0.96689

=== Fold 3 starting ===
  XGBoost done in 27.1s | AUC: 0.96798


...

  CatBoost done in 291.9s | AUC: 0.96710

=== Fold 4 starting ===
  XGBoost done in 30.2s | AUC: 0.96834


Default metric period is 5 because AUC is/are not implemented for GPU


  CatBoost done in 266.7s | AUC: 0.96757

=== Fold 5 starting ===
  XGBoost done in 27.5s | AUC: 0.96755


Default metric period is 5 because AUC is/are not implemented for GPU


  CatBoost done in 284.8s | AUC: 0.96668

=== Individual Final OOF AUCs ===
LightGBM: 0.96766
XGBoost:  0.96773
CatBoost: 0.96687

=== FINAL STACKED OOF AUC: 0.96777 ===
vs LightGBM alone: 0.96766

Saved: submission_v2.csv
       id  addicted_label
0  691369        0.974678
1  691370        0.963605
2  691371        0.935455
3  691372        0.973772
4  691373        0.974608
Shape: (296302, 2)


In [31]:
# ============ Cell: V3 — Multi-Seed Bagging + Pseudo-Labeling ============
import time

# ---- Step 1: Pseudo-labeling from current best predictions ----
CONFIDENCE_HIGH = 0.97
CONFIDENCE_LOW = 0.03

confident_mask = (final_test_predictions >= CONFIDENCE_HIGH) | (final_test_predictions <= CONFIDENCE_LOW)
print(f"Confident test rows: {confident_mask.sum()} of {len(final_test_predictions)} ({confident_mask.mean()*100:.1f}%)")

pseudo_X = X_test[confident_mask].copy()
pseudo_y = (final_test_predictions[confident_mask] >= 0.5).astype(int)
pseudo_y = pd.Series(pseudo_y, index=pseudo_X.index)

# Combine real training data + pseudo-labeled data
X_aug = pd.concat([X, pseudo_X], axis=0).reset_index(drop=True)
y_aug = pd.concat([y, pseudo_y], axis=0).reset_index(drop=True)
print(f"Augmented training size: {len(X_aug)} (original: {len(X)})")

X_cat_aug = X_aug.copy()
X_cat_aug['gender'] = X_cat_aug['gender'].astype(str)
cat_idx_aug = [X_cat_aug.columns.get_loc('gender')]

# ---- Step 2: Multi-seed bagged LightGBM on augmented data ----
SEEDS_BAG = [42, 123, 777]
oof_bag = np.zeros(len(X))  # OOF still measured only on real data
test_bag = np.zeros(len(X_test))

for seed_i, bag_seed in enumerate(SEEDS_BAG, start=1):
    print(f"\n########## Seed {bag_seed} ({seed_i}/{len(SEEDS_BAG)}) ##########", flush=True)

    lgb_params_bag = dict(
        objective='binary', metric='auc', boosting_type='gbdt',
        n_estimators=4000, learning_rate=0.02, num_leaves=255,
        min_child_samples=20, subsample=0.8, subsample_freq=1,
        colsample_bytree=0.75, reg_alpha=0.5, reg_lambda=1.0,
        verbosity=-1, random_state=bag_seed,
        device_type='gpu', gpu_use_dp=True
    )

    skf_bag = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=bag_seed)

    # Train on augmented data, but only validate/predict OOF on real rows
    real_idx_in_aug = np.arange(len(X))

    for fold, (tr_idx, val_idx) in enumerate(skf_bag.split(X_aug, y_aug), start=1):
        # Ensure validation fold only contains REAL rows (never pseudo-labeled ones)
        val_idx_real = val_idx[val_idx < len(X)]
        if len(val_idx_real) == 0:
            continue

        t0 = time.time()
        m = lgb.LGBMClassifier(**lgb_params_bag)
        m.fit(X_aug.iloc[tr_idx], y_aug.iloc[tr_idx],
              eval_set=[(X_aug.iloc[val_idx_real], y_aug.iloc[val_idx_real])],
              callbacks=[lgb.early_stopping(200, verbose=False), lgb.log_evaluation(0)])

        oof_bag[val_idx_real] += m.predict_proba(X_aug.iloc[val_idx_real])[:, 1] / len(SEEDS_BAG)
        test_bag += m.predict_proba(X_test)[:, 1] / (N_FOLDS * len(SEEDS_BAG))
        print(f"  Seed {bag_seed} Fold {fold} done in {time.time()-t0:.1f}s | AUC: {roc_auc_score(y_aug.iloc[val_idx_real], m.predict_proba(X_aug.iloc[val_idx_real])[:, 1]):.5f}", flush=True)

print(f"\n=== V3 Bagged+Pseudo-labeled OOF AUC: {roc_auc_score(y, oof_bag):.5f} ===")
print(f"vs V2 stack: 0.96777")
print(f"vs V2 LightGBM alone: 0.96766")

# ---- Step 3: Blend V3 with V2 stack (safety net — don't fully replace, combine) ----
final_v3_predictions = (test_bag * 0.6) + (final_test_predictions * 0.4)

submission_v3 = pd.DataFrame({
    'id': test['id'],
    'addicted_label': final_v3_predictions
})
submission_v3.to_csv('submission_v3.csv', index=False)
print("\nSaved: submission_v3.csv")
print(submission_v3.head())

Confident test rows: 141943 of 296302 (47.9%)
Augmented training size: 833312 (original: 691369)

########## Seed 42 (1/3) ##########
  Seed 42 Fold 1 done in 244.8s | AUC: 0.96701
  Seed 42 Fold 2 done in 272.2s | AUC: 0.96821
  Seed 42 Fold 3 done in 213.9s | AUC: 0.96729
  Seed 42 Fold 4 done in 219.8s | AUC: 0.96844
  Seed 42 Fold 5 done in 231.8s | AUC: 0.96771

########## Seed 123 (2/3) ##########
  Seed 123 Fold 1 done in 220.1s | AUC: 0.96743
  Seed 123 Fold 2 done in 229.6s | AUC: 0.96772
  Seed 123 Fold 3 done in 233.0s | AUC: 0.96787
  Seed 123 Fold 4 done in 245.0s | AUC: 0.96823
  Seed 123 Fold 5 done in 202.4s | AUC: 0.96735

########## Seed 777 (3/3) ##########
  Seed 777 Fold 1 done in 259.7s | AUC: 0.96696
  Seed 777 Fold 2 done in 218.8s | AUC: 0.96760
  Seed 777 Fold 3 done in 231.1s | AUC: 0.96822
  Seed 777 Fold 4 done in 193.1s | AUC: 0.96797
  Seed 777 Fold 5 done in 252.5s | AUC: 0.96768

=== V3 Bagged+Pseudo-labeled OOF AUC: 0.96799 ===
vs V2 stack: 0.96777
vs 

In [34]:
# Add V3 bagged score
model_scores['lightgbm_v3_bagged'] = roc_auc_score(y, oof_bag)
print("saved:", model_scores['lightgbm_v3_bagged'])

saved: 0.967990778607162
